# Projet : Estimation de la valeur d'option — Modèle de Cox, Ross et Rubinstein

**Cours de Mathématiques Financières**

---

*Notebook réalisé dans le cadre d'un projet universitaire sur la valorisation d'options par le modèle binomial de Cox-Ross-Rubinstein (CRR).*

## Partie 1 — Introduction du projet

### 1.1 Contexte et objectif

Ce projet porte sur l'estimation du prix d'options financières à l'aide du modèle discret de **Cox, Ross et Rubinstein (CRR)**, introduit en 1979. L'objectif est de mettre en œuvre ce modèle sur des données de marché réelles (indice CAC 40, Brent, etc.), d'en calibrer les paramètres, puis d'analyser ses performances et ses limites.

### 1.2 Qu'est-ce qu'une option financière ?

Une **option financière** est un contrat dérivé qui confère à son détenteur le droit — mais non l'obligation — d'acheter (**call**) ou de vendre (**put**) un actif sous-jacent à un prix convenu à l'avance, appelé **strike** $K$, à une date donnée appelée **maturité** $T$.

Le **payoff** (gain) à maturité est défini par :

$$
	ext{Payoff}_{	ext{call}} = \max(S_T - K,\ 0)
$$
$$
	ext{Payoff}_{	ext{put}} = \max(K - S_T,\ 0)
$$

où $S_T$ désigne le prix du sous-jacent à maturité.

### 1.3 Le modèle de Cox-Ross-Rubinstein

Le modèle CRR repose sur l'hypothèse que, sur chaque intervalle de temps $\Delta t$, le prix du sous-jacent peut :
- **monter** d'un facteur $u > 1$,
- **baisser** d'un facteur $d < 1$, avec $d = 1/u$.

Cette dynamique discrète permet de construire un **arbre binomial** représentant l'ensemble des trajectoires possibles du sous-jacent jusqu'à la maturité.

### 1.4 Pourquoi un arbre binomial ?

L'arbre binomial constitue une approximation discrète du mouvement brownien géométrique utilisé dans le modèle de Black-Scholes. Lorsque le nombre d'étapes $n$ tend vers l'infini, le prix CRR converge vers le prix de Black-Scholes. L'arbre est particulièrement adapté à la valorisation des **options américaines**, où l'exercice anticipé est possible à chaque nœud.

### 1.5 Confrontation au marché réel

L'intérêt principal de ce projet réside dans la **confrontation du modèle théorique aux données réelles**. Nous allons :
1. Estimer la volatilité historique de l'actif à partir de ses rendements passés ;
2. Calibrer les paramètres $u$, $d$ et $p^*$ du modèle ;
3. Calculer le prix théorique d'une option ;
4. Analyser la sensibilité du modèle et sa stabilité temporelle par backtest.

## Partie 2 — Importation des bibliothèques et des modules

On importe les bibliothèques Python standard ainsi que les modules du projet. Chaque module a un rôle bien défini :

| Module | Rôle |
|---|---|
| `market_data` | Téléchargement et préparation des données de marché |
| `option` | Définition d'une option financière |
| `calibration` | Estimation des paramètres du modèle CRR |
| `crr` | Construction de l'arbre binomial et pricing |
| `backtester` | Évaluation empirique du modèle dans le temps |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Modules du projet
from market_data import build_dataset, get_S0_and_log_returns

from option import Option

from calibration import (
    estimate_volatility,
    calibrate_crr_parameters,
    volatility_confidence_interval,
    compute_ud,
    compute_risk_neutral_probability
)

from crr import (
    crr_price,
    build_price_tree,
    extract_tree_levels
)

from backtester import (
    run_backtest,
    summarize_backtest_results,
    analyze_stability
)

print("Imports réussis.")

## Partie 3 — Chargement des données de marché

On utilise la fonction `build_dataset` du module `market_data.py` pour télécharger et préparer les données. Le ticker peut être facilement modifié pour étudier un autre actif.

La plage temporelle choisie (2020–2025) couvre des périodes de forte volatilité (crise Covid-19 de 2020, hausse des taux en 2022) ce qui enrichit l'analyse.

In [ ]:
ticker = "^FCHI"   # CAC 40
start_date = "2020-01-01"
end_date = "2025-01-01"

# Exemples d'autres actifs disponibles :
# ticker = "BZ=F"    # Brent Crude Oil
# ticker = "MC.PA"   # LVMH
# ticker = "TTE.PA"  # TotalEnergies
# ticker = "^GSPC"   # S&P 500

data = build_dataset(ticker, start_date, end_date)
data.head()

## Partie 4 — Analyse descriptive des données

On commence par examiner les statistiques descriptives de la série, puis on visualise les trois indicateurs clés : le prix de clôture, les rendements logarithmiques et la volatilité glissante.

In [ ]:
print("Nombre d'observations :", len(data))
print("Colonnes disponibles  :", data.columns.tolist())
data.describe()

In [ ]:
# Graphique 1 — Prix de clôture
plt.figure(figsize=(10, 5))
plt.plot(data.index, data["Close"], color="#1f77b4", linewidth=1)
plt.title(f"Évolution du prix de clôture — {ticker}", fontsize=13)
plt.xlabel("Date")
plt.ylabel("Prix de clôture")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Graphique 2 — Rendements logarithmiques
plt.figure(figsize=(10, 5))
plt.plot(data.index, data["log_return"], color="#ff7f0e", linewidth=0.8, alpha=0.8)
plt.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.title(f"Rendements logarithmiques journaliers — {ticker}", fontsize=13)
plt.xlabel("Date")
plt.ylabel("Rendement logarithmique")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Graphique 3 — Volatilité glissante annualisée (fenêtre 21 jours)
plt.figure(figsize=(10, 5))
plt.plot(data.index, data["rolling_vol_21"], color="#2ca02c", linewidth=1)
plt.title(f"Volatilité glissante annualisée (21 jours) — {ticker}", fontsize=13)
plt.xlabel("Date")
plt.ylabel("Volatilité annualisée")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### Commentaire des graphiques

- **Prix de clôture** : On observe une chute brutale début 2020 lors de la crise sanitaire, suivie d'un rebond progressif. Les années 2021–2022 montrent une forte oscillation liée aux tensions inflationnistes et à la remontée des taux directeurs.

- **Rendements logarithmiques** : La série présente une forte hétéroscédasticité — les phases de turbulence (Mars 2020, fin 2022) se distinguent nettement par des rendements extrêmes, positifs ou négatifs. Ce comportement est cohérent avec l'hypothèse de **volatilité stochastique** observée sur les marchés réels.

- **Volatilité glissante** : Le pic de volatilité de Mars 2020 (supérieur à 50 % en annualisé) illustre clairement le choc Covid. La volatilité retombe ensuite progressivement avant de se raviver lors des épisodes d'incertitude macroéconomique ultérieurs.

## Partie 5 — Extraction du prix initial et des rendements logarithmiques

La fonction `get_S0_and_log_returns` extrait :
- **$S_0$** : le dernier prix de clôture observé, qui sert de point de départ à l'arbre binomial ;
- **les rendements logarithmiques** : utilisés pour estimer la volatilité historique.

In [ ]:
S0, log_returns = get_S0_and_log_returns(data)

print(f"Prix initial S0           : {S0:.4f}")
print(f"Nombre de rendements log  : {len(log_returns)}")
print(f"Rendement moyen journalier: {log_returns.mean():.6f}")
print(f"Écart-type journalier     : {log_returns.std():.6f}")

**Remarque** : $S_0$ correspond au **dernier prix de clôture** disponible dans la série. Les rendements logarithmiques $r_t = \ln(S_t / S_{t-1})$ permettent d'estimer l'écart-type empirique $\hat{\sigma}_{	ext{jour}}$, qui est ensuite annualisé pour obtenir la volatilité $\sigma$ utilisée dans le modèle CRR.

## Partie 6 — Définition de l'option étudiée

On définit une option **call européenne at-the-money (ATM)** : le strike est égal au prix spot $S_0$, ce qui correspond à la situation où l'option n'est ni dans la monnaie ni hors de la monnaie. C'est le cas le plus courant dans l'analyse de sensibilité.

In [ ]:
strike = S0          # option at-the-money
maturity = 1.0       # maturité d'un an
option_type = "call" # option call

option = Option(
    strike=strike,
    maturity=maturity,
    option_type=option_type
)

print(option)
print(f"\nPayoff si S_T = {S0 * 1.1:.2f} (hausse 10 %) : {option.payoff(S0 * 1.1):.4f}")
print(f"Payoff si S_T = {S0 * 0.9:.2f} (baisse 10 %) : {option.payoff(S0 * 0.9):.4f}")

# Pour tester un put, décommenter la ligne suivante :
# option = Option(strike=S0, maturity=1.0, option_type="put")

## Partie 7 — Calibration des paramètres du modèle CRR

### 7.1 Formules théoriques

La **volatilité historique annualisée** est estimée par :

$$
\sigma = \hat{s}(r_t) \times \sqrt{252}
$$

où $\hat{s}(r_t)$ est l'écart-type empirique des rendements logarithmiques journaliers (estimateur corrigé, divisé par $n-1$).

Les **facteurs de hausse et de baisse** du modèle CRR sont définis par :

$$
u = e^{\sigma \sqrt{\Delta t}}, \quad d = e^{-\sigma \sqrt{\Delta t}} = \frac{1}{u}
$$

La **probabilité risque-neutre** est donnée par :

$$
p^* = \frac{e^{r \Delta t} - d}{u - d}
$$

Pour que le modèle soit arbitrage-free, il faut vérifier : $0 \leq p^* \leq 1$, soit $d < e^{r \Delta t} < u$.

### 7.2 Calibration

In [ ]:
n_steps = 100
risk_free_rate = 0.03  # taux sans risque fixé à 3 % par an

params = calibrate_crr_parameters(
    log_returns=log_returns,
    maturity=option.maturity,
    n_steps=n_steps,
    risk_free_rate=risk_free_rate
)

print("=" * 45)
print("  Paramètres calibrés du modèle CRR")
print("=" * 45)
print(f"  Volatilité annualisée σ : {params['sigma']:.4f}  ({params['sigma']*100:.2f} %)")
print(f"  Taux sans risque r      : {params['r']:.4f}  ({params['r']*100:.2f} %)")
print(f"  Pas de temps Δt         : {params['dt']:.6f}")
print(f"  Facteur de hausse u     : {params['u']:.6f}")
print(f"  Facteur de baisse d     : {params['d']:.6f}")
print(f"  Probabilité risque-neutre p* : {params['p_star']:.6f}")
print("=" * 45)

# Vérification de la condition d'absence d'arbitrage
assert 0 < params['p_star'] < 1, "Condition d'absence d'arbitrage non vérifiée !"
print("\n✓ Condition d'absence d'arbitrage vérifiée : 0 < p* < 1")

## Partie 8 — Intervalle de confiance pour la volatilité

### Fondement statistique

La volatilité $\sigma$ est estimée à partir d'un **échantillon fini** de $n$ rendements. Cette estimation est incertaine. Sous l'hypothèse de normalité des rendements log, la statistique :

$$
\frac{(n-1) \hat{s}^2}{\sigma^2} \sim \chi^2(n-1)
$$

permet de construire un intervalle de confiance pour la variance, puis pour la volatilité annualisée.

Un intervalle de confiance à $(1-\alpha)$ % pour $\sigma$ s'écrit :

$$
\left[ \sqrt{\frac{(n-1)\hat{s}^2}{\chi^2_{1-\alpha/2}}} \times \sqrt{252},\ \sqrt{\frac{(n-1)\hat{s}^2}{\chi^2_{\alpha/2}}} \times \sqrt{252} \right]
$$

In [ ]:
ci_low, ci_high = volatility_confidence_interval(log_returns, confidence_level=0.95)

print(f"Volatilité estimée σ̂         : {params['sigma']:.4f}  ({params['sigma']*100:.2f} %)")
print(f"Borne inférieure IC à 95 %   : {ci_low:.4f}  ({ci_low*100:.2f} %)")
print(f"Borne supérieure IC à 95 %   : {ci_high:.4f}  ({ci_high*100:.2f} %)")
print(f"\nLargeur de l'intervalle      : {(ci_high - ci_low):.4f}")

# Visualisation de l'intervalle
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(["Volatilité"], [ci_high - ci_low], left=ci_low, color="#aec7e8", edgecolor="#1f77b4", height=0.3)
ax.axvline(params['sigma'], color="#1f77b4", linewidth=2, label=f"σ̂ = {params['sigma']:.4f}")
ax.set_xlabel("Volatilité annualisée")
ax.set_title("Intervalle de confiance à 95 % pour la volatilité")
ax.legend()
plt.tight_layout()
plt.show()

**Interprétation** : Plus la taille de l'échantillon est grande, plus l'intervalle est étroit. La largeur de cet intervalle reflète l'**incertitude statistique** sur l'estimation de la volatilité. Dans le contexte du pricing d'options, cette incertitude se répercute directement sur le prix théorique — un motif supplémentaire pour réaliser l'analyse de sensibilité à la volatilité présentée en Partie 11.

## Partie 9 — Calcul du prix théorique de l'option avec le modèle CRR

### Principe de la rétropropagation

Le modèle CRR opère en deux phases :
1. **Construction de l'arbre de prix** du sous-jacent : à chaque étape $i$, le prix peut monter (×$u$) ou baisser (×$d$).
2. **Rétropropagation** : à partir des payoffs terminaux, on remonte l'arbre en actualisant les valeurs espérées sous la mesure risque-neutre :

$$
V_i^j = e^{-r \Delta t} \left[ p^* V_{i+1}^{j+1} + (1 - p^*) V_{i+1}^j ight]
$$

Le prix de l'option est la valeur obtenue au nœud racine $V_0^0$.

In [ ]:
price, value_tree = crr_price(
    S0=S0,
    option=option,
    u=params["u"],
    d=params["d"],
    r=params["r"],
    dt=params["dt"],
    p_star=params["p_star"],
    american=False
)

print("=" * 50)
print(f"  Prix CRR de l'option (européenne) : {price:.4f}")
print("=" * 50)
print(f"\n  Paramètres utilisés :")
print(f"    S0     = {S0:.2f}")
print(f"    Strike = {option.strike:.2f}  (ATM)")
print(f"    T      = {option.maturity} an")
print(f"    σ      = {params['sigma']:.4f}")
print(f"    r      = {params['r']:.4f}")
print(f"    n      = {n_steps} étapes")

In [ ]:
# Prix d'une option américaine (exercice anticipé possible)
price_am, _ = crr_price(
    S0=S0,
    option=option,
    u=params["u"],
    d=params["d"],
    r=params["r"],
    dt=params["dt"],
    p_star=params["p_star"],
    american=True
)

print(f"Prix CRR option européenne : {price:.4f}")
print(f"Prix CRR option américaine : {price_am:.4f}")
print(f"Prime d'exercice anticipé  : {price_am - price:.4f}")

## Partie 10 — Visualisation pédagogique de l'arbre binomial

Pour des raisons de lisibilité, on construit ici un petit arbre avec seulement **5 étapes**. En pratique, on utilise 100 étapes ou plus pour obtenir un prix précis.

In [ ]:
small_n_steps = 5

small_dt = option.maturity / small_n_steps
small_u, small_d = compute_ud(params["sigma"], small_dt)
small_p_star = compute_risk_neutral_probability(
    params["r"], small_dt, small_u, small_d
)

print(f"Paramètres pour l'arbre à {small_n_steps} étapes :")
print(f"  Δt    = {small_dt:.4f}")
print(f"  u     = {small_u:.4f}")
print(f"  d     = {small_d:.4f}")
print(f"  p*    = {small_p_star:.4f}")

price_tree = build_price_tree(S0=S0, u=small_u, d=small_d, n_steps=small_n_steps)

# Affichage sous forme de DataFrame
tree_df = pd.DataFrame(
    [pd.Series(level) for level in price_tree]
)
tree_df.index.name = "Étape"
tree_df.columns = [f"Nœud {j}" for j in tree_df.columns]
tree_df.round(2)

In [ ]:
# Représentation graphique de l'arbre (5 étapes)
fig, ax = plt.subplots(figsize=(10, 6))

for i, level in enumerate(price_tree):
    for j, price_node in enumerate(level):
        x = i
        y = j - i / 2  # centrage vertical
        ax.plot(x, y, 'o', color='#1f77b4', markersize=8)
        ax.text(x + 0.05, y + 0.05, f"{price_node:.0f}", fontsize=7, ha='left')
        # Liaisons vers l'étape suivante
        if i < small_n_steps:
            next_level = price_tree[i + 1]
            # branche hausse
            y_up = (j + 1) - (i + 1) / 2
            ax.plot([x, x + 1], [y, y_up], '-', color='#2ca02c', linewidth=0.8, alpha=0.6)
            # branche baisse
            y_dn = j - (i + 1) / 2
            ax.plot([x, x + 1], [y, y_dn], '-', color='#d62728', linewidth=0.8, alpha=0.6)

ax.set_title(f"Arbre binomial CRR — {small_n_steps} étapes (prix du sous-jacent)", fontsize=12)
ax.set_xlabel("Étape")
ax.set_ylabel("Nœud (centré)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Lecture de l'arbre** : Chaque nœud $(i, j)$ représente le prix $S_0 \cdot u^j \cdot d^{i-j}$ après $i$ périodes, avec $j$ hausses et $(i-j)$ baisses. L'arbre est **recombinable** : un chemin hausse-baisse mène au même nœud qu'un chemin baisse-hausse, ce qui réduit la complexité de $2^n$ à $\mathcal{O}(n^2)$.

## Partie 11 — Analyse de sensibilité

L'analyse de sensibilité permet d'évaluer l'impact de chaque paramètre sur le prix théorique de l'option. Elle est essentielle pour comprendre les **risques de modèle** (_model risk_).

### 11.1 Sensibilité au nombre d'étapes $n$

Le nombre d'étapes $n$ contrôle la **granularité de l'arbre**. Pour $n 	o \infty$, le modèle CRR converge vers le prix de Black-Scholes.

In [ ]:
steps_list = [5, 10, 25, 50, 100, 200, 500]
prices_by_steps = []

for steps in steps_list:
    params_steps = calibrate_crr_parameters(
        log_returns=log_returns,
        maturity=option.maturity,
        n_steps=steps,
        risk_free_rate=risk_free_rate
    )
    price_steps, _ = crr_price(
        S0=S0,
        option=option,
        u=params_steps["u"],
        d=params_steps["d"],
        r=params_steps["r"],
        dt=params_steps["dt"],
        p_star=params_steps["p_star"],
        american=False
    )
    prices_by_steps.append(price_steps)

sensitivity_steps = pd.DataFrame({"n_steps": steps_list, "option_price": prices_by_steps})
print(sensitivity_steps.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sensitivity_steps["n_steps"], sensitivity_steps["option_price"],
         marker="o", color="#1f77b4")
plt.axhline(sensitivity_steps["option_price"].iloc[-1], color="gray",
            linestyle="--", linewidth=0.8, label=f"Asymptote ≈ {sensitivity_steps['option_price'].iloc[-1]:.2f}")
plt.title("Sensibilité du prix de l'option au nombre d'étapes $n$", fontsize=12)
plt.xlabel("Nombre d'étapes $n$")
plt.ylabel("Prix de l'option")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**Analyse** : Le prix de l'option présente une légère oscillation pour les faibles valeurs de $n$, puis se stabilise rapidement. Ce phénomène de convergence oscilante est caractéristique du modèle CRR : les prix pour $n$ pair et impair convergent alternativement vers la limite. Au-delà de 100 étapes, la précision numérique est généralement suffisante pour les applications pratiques.

### 11.2 Sensibilité au strike $K$

Le strike détermine si l'option est **dans la monnaie** (_in-the-money_), **à la monnaie** (_at-the-money_) ou **hors de la monnaie** (_out-of-the-money_).

In [ ]:
strike_values = [0.8 * S0, 0.9 * S0, S0, 1.1 * S0, 1.2 * S0]
prices_by_strike = []

for K in strike_values:
    opt_K = Option(strike=K, maturity=maturity, option_type=option_type)
    price_K, _ = crr_price(
        S0=S0, option=opt_K,
        u=params["u"], d=params["d"],
        r=params["r"], dt=params["dt"],
        p_star=params["p_star"], american=False
    )
    prices_by_strike.append(price_K)

sensitivity_strike = pd.DataFrame({
    "strike": [round(k, 2) for k in strike_values],
    "moneyness": ["ITM -20%", "ITM -10%", "ATM", "OTM +10%", "OTM +20%"],
    "option_price": prices_by_strike
})
print(sensitivity_strike.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sensitivity_strike["strike"], sensitivity_strike["option_price"],
         marker="o", color="#2ca02c")
plt.axvline(S0, color="gray", linestyle="--", linewidth=0.8, label="ATM (K = S₀)")
plt.title("Sensibilité du prix du call au strike $K$", fontsize=12)
plt.xlabel("Strike $K$")
plt.ylabel("Prix de l'option (call)")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**Analyse** : Pour un call, le prix décroît de manière monotone avec le strike — plus le strike est élevé, moins l'option a de valeur intrinsèque. La relation est convexe, cohérente avec le profil de payoff $\max(S_T - K, 0)$. Une option ITM (strike bas) vaut davantage car elle a une probabilité plus élevée de générer un payoff positif.

### 11.3 Sensibilité à la volatilité $\sigma$

La volatilité est le paramètre le plus sensible dans la valorisation des options. Elle est liée au **vega** de l'option :

$$
\text{Vega} = \frac{\partial V}{\partial \sigma} > 0
$$

Une hausse de volatilité accroît l'incertitude sur $S_T$ et augmente donc la valeur des options (calls comme puts).

In [ ]:
sigma_values = np.linspace(0.5 * params["sigma"], 1.5 * params["sigma"], 20)
prices_by_sigma = []

for sigma in sigma_values:
    u_s, d_s = compute_ud(sigma, params["dt"])
    p_s = compute_risk_neutral_probability(params["r"], params["dt"], u_s, d_s)
    price_s, _ = crr_price(
        S0=S0, option=option,
        u=u_s, d=d_s, r=params["r"],
        dt=params["dt"], p_star=p_s, american=False
    )
    prices_by_sigma.append(price_s)

sensitivity_sigma = pd.DataFrame({
    "sigma": sigma_values,
    "option_price": prices_by_sigma
})

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sensitivity_sigma["sigma"], sensitivity_sigma["option_price"],
         marker="o", color="#d62728")
plt.axvline(params["sigma"], color="gray", linestyle="--", linewidth=0.8,
            label=f"σ calibré = {params['sigma']:.4f}")
plt.title("Sensibilité du prix de l'option à la volatilité $\sigma$", fontsize=12)
plt.xlabel("Volatilité annualisée $\sigma$")
plt.ylabel("Prix de l'option")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**Analyse** : La relation entre volatilité et prix de l'option est strictement croissante et légèrement convexe. C'est une propriété fondamentale : une option est en quelque sorte un pari sur l'incertitude future. Plus cette incertitude est grande ($\sigma$ élevé), plus la valeur de l'option augmente. Ce résultat souligne l'importance d'une estimation précise de la volatilité — point directement lié à l'intervalle de confiance construit en Partie 8.

## Partie 12 — Backtest du modèle dans le temps

Le backtest consiste à **appliquer le modèle de manière répétée** sur l'ensemble de la série historique, en recalibrant la volatilité à chaque date sur une **fenêtre glissante de 252 jours** (soit environ un an de trading). Cela permet d'observer la stabilité et la cohérence du modèle dans le temps.

**Protocole** : pour chaque date $t$ disponible après la première fenêtre de calibration, on :
1. Calibre $\sigma$ sur les 252 rendements précédents ;
2. Calcule les paramètres $u$, $d$, $p^*$ ;
3. Calcule le prix théorique de l'option pour $S_0 = S_t$.

> ⚠️ En l'absence de vrais prix d'options observés, ce backtest mesure la **cohérence interne** du modèle plutôt que sa capacité prédictive absolue.

In [ ]:
results = run_backtest(
    data=data,
    option=option,
    pricing_function=crr_price,
    calibration_function=calibrate_crr_parameters,
    n_steps=100,
    risk_free_rate=risk_free_rate,
    window=252
)

print(f"Nombre de dates de test : {len(results)}")
print(f"Période : {results['date'].min().date()} → {results['date'].max().date()}")
results.head()

In [ ]:
results.tail()

In [ ]:
summary = summarize_backtest_results(results)

print("=" * 45)
print("  Résumé du backtest")
print("=" * 45)
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k:<30} : {v:.4f}")
    else:
        print(f"  {k:<30} : {v}")
print("=" * 45)

In [ ]:
stability = analyze_stability(results)

print("Statistiques de volatilité calibrée :")
for k, v in stability["sigma_stats"].items():
    print(f"  {k:<20} : {v:.4f}")

print("\nStatistiques de p* :")
for k, v in stability["p_star_stats"].items():
    print(f"  {k:<20} : {v:.6f}")

In [ ]:
# 12.1 — Prix prédit de l'option dans le temps
plt.figure(figsize=(10, 5))
plt.plot(results["date"], results["prix_prédit"], color="#1f77b4", linewidth=1)
plt.title("Prix théorique CRR de l'option dans le temps", fontsize=12)
plt.xlabel("Date")
plt.ylabel("Prix prédit")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# 12.2 — Volatilité calibrée dans le temps
plt.figure(figsize=(10, 5))
plt.plot(results["date"], results["sigma"], color="#2ca02c", linewidth=1)
plt.title("Volatilité calibrée dans le temps (fenêtre glissante 252 j)", fontsize=12)
plt.xlabel("Date")
plt.ylabel("Volatilité annualisée σ")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# 12.3 — Probabilité risque-neutre p* dans le temps
plt.figure(figsize=(10, 5))
plt.plot(results["date"], results["p_star"], color="#9467bd", linewidth=1)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="p* = 0.5")
plt.title("Probabilité risque-neutre $p^*$ dans le temps", fontsize=12)
plt.xlabel("Date")
plt.ylabel("$p^*$")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# 12.4 — Prix prédit et moyenne glissante 21 jours
serie_temporelle = stability["serie_temporelle"]

plt.figure(figsize=(10, 5))
plt.plot(serie_temporelle["date"], serie_temporelle["prix_prédit"],
         color="#1f77b4", linewidth=0.8, alpha=0.7, label="Prix prédit")
plt.plot(serie_temporelle["date"], serie_temporelle["rolling_mean_prix"],
         color="#d62728", linewidth=1.5, label="Moyenne glissante 21 jours")
plt.title("Prix théorique de l'option et moyenne glissante", fontsize=12)
plt.xlabel("Date")
plt.ylabel("Prix")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### Analyse des résultats du backtest

- **Prix prédit** : Le prix théorique de l'option évolue avec le prix du sous-jacent $S_t$ et la volatilité recalibrée $\sigma_t$. Les pics de prix correspondent aux périodes de forte volatilité (Mars 2020), où l'incertitude augmente la valeur de l'option.

- **Volatilité recalibrée** : L'estimation glissante de $\sigma$ capture bien les régimes de volatilité. On observe que la volatilité n'est pas constante dans le temps, ce qui est une **limite fondamentale** du modèle CRR dans sa version de base.

- **Probabilité risque-neutre $p^*$** : Elle reste proche de 0.5, avec de légères variations liées aux fluctuations de $\sigma$ et du taux sans risque. Sa stabilité est un signe de cohérence du modèle.

- **Moyenne glissante** : Elle lisse les fluctuations à court terme et met en évidence les tendances structurelles du prix théorique sur la période.

## Partie 13 — Discussion critique sur la validité du modèle

### 13.1 Hypothèses simplificatrices

Le modèle de Cox-Ross-Rubinstein est fondé sur plusieurs hypothèses qui constituent autant de **sources d'écart avec les marchés réels** :

**1. Volatilité constante.** Dans le modèle utilisé, $\sigma$ est estimée historiquement et supposée constante sur tout l'horizon de valorisation. Or, les marchés financiers présentent un phénomène bien documenté de **volatility clustering** : les périodes de forte volatilité se regroupent temporellement (visible sur le graphique de la Partie 4). Des modèles à volatilité stochastique (Heston, SABR) permettent de mieux capturer cette dynamique.

**2. Taux sans risque fixe.** Le taux $r = 3 \%$ est fixé manuellement pour l'ensemble de la période. En pratique, le taux sans risque est une courbe de taux dépendant de la maturité, de la devise et des conditions de marché (OIS, OAT, Treasuries). Une structure par terme des taux affinerait l'estimation.

**3. Distribution des rendements.** Le modèle CRR, comme Black-Scholes, suppose que les rendements logarithmiques suivent une **loi normale**. Les données réelles présentent pourtant des **queues épaisses** (_fat tails_) et une **asymétrie négative** (_skewness_). Les événements extrêmes sont plus fréquents que ce que la loi normale prédit, ce que la **volatilité de crise de Mars 2020** illustre parfaitement.

**4. Marché complet et absence de frictions.** Le modèle suppose un marché parfait : pas de coûts de transaction, de bid-ask spread, de contraintes de liquidité, ni de dividendes discrets. Ces éléments peuvent avoir un impact significatif sur le prix réel des options.

**5. Absence de prix d'options observés pour la validation.** Dans ce projet, aucun prix d'options de marché n'est directement disponible. Le backtest mesure donc la **cohérence interne** du modèle (stabilité des paramètres, comportement dans le temps) plutôt qu'une erreur de pricing absolue.

### 13.2 Points forts du modèle

Malgré ces limites, le modèle CRR conserve une valeur pédagogique et pratique importante :

- **Transparence et interprétabilité** : l'arbre binomial est visuellement lisible et chaque nœud a une interprétation économique claire.
- **Fondement rigoureux** : le pricing par absence d'arbitrage et la mesure risque-neutre constituent les fondations de la finance quantitative moderne.
- **Convergence vers Black-Scholes** : le modèle CRR est une approximation discrète contrôlée du modèle log-normal continu.
- **Flexibilité** : la rétropropagation permet de traiter naturellement les options américaines et les options à barrière.

### 13.3 Pistes d'amélioration

Pour aller au-delà de cette première version, on pourrait :
- Utiliser une **volatilité implicite** extraite des prix d'options observés (_implied volatility_) plutôt qu'une volatilité historique ;
- Introduire un modèle à **volatilité stochastique** pour modéliser le smile de volatilité ;
- Intégrer des **dividendes** si l'actif sous-jacent en verse (cas de LVMH, TotalEnergies, etc.) ;
- Comparer les prix CRR aux prix Black-Scholes comme _benchmark_ théorique.

## Partie 14 — Conclusion

Ce projet a permis de mettre en œuvre le modèle de **Cox-Ross-Rubinstein** de manière complète, de la collecte des données à l'évaluation empirique, en passant par la calibration et l'analyse de sensibilité.

**Ce qui a été réalisé :**

1. **Données réelles** : les données de l'indice CAC 40 ont été téléchargées, nettoyées et transformées en rendements logarithmiques. La volatilité glissante a mis en évidence les régimes de marché successifs sur la période 2020–2025.

2. **Calibration** : la volatilité historique annualisée a été estimée et un intervalle de confiance à 95 % a été construit, quantifiant l'incertitude statistique inhérente à cette estimation.

3. **Pricing** : le prix théorique d'une option call européenne at-the-money a été calculé par construction d'un arbre binomial à 100 étapes et rétropropagation sous la mesure risque-neutre.

4. **Sensibilité** : les analyses montrent que le prix de l'option est croissant avec la volatilité (vega positif) et décroissant avec le strike (delta positif), conformément à la théorie. La convergence numérique est rapide dès $n \geq 50$ étapes.

5. **Backtest** : le modèle a été appliqué de façon systématique sur l'ensemble de la série, révélant la non-constance de la volatilité dans le temps — une limite bien identifiée du modèle dans sa version de base.

**Bilan critique :**

Le modèle CRR est **mathématiquement cohérent** et **pédagogiquement pertinent**. Il illustre les concepts fondamentaux de la finance quantitative : absence d'arbitrage, mesure risque-neutre, actualisation des flux futurs. Ses limites — volatilité constante, distribution normale des rendements, absence de frictions de marché — sont connues et documentées. Des extensions (volatilité stochastique, smile de volatilité, modèles à sauts) permettraient de se rapprocher davantage de la réalité des marchés.

---

*Fin du notebook — Projet CRR, Mathématiques Financières*